In [4]:
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np
from sklearn.cluster import DBSCAN

In [5]:
TEST_DIR = "../datasets/images/val"
TEST_LABEL_DIR = "../runs/crowd_yolov5s_detect/labels"
RESULT_SAVE_DIR = "../image_results/20251014_v1"

In [6]:
def label_path_for_image(img_path):
    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl = os.path.join(TEST_LABEL_DIR, base + '.txt')
    return lbl if os.path.exists(lbl) else None

def parse_yolo_label(lbl_path):
    if not lbl_path:
        return None
    boxes = []
    try:
        with open(lbl_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls = int(float(parts[0]))
                x, y, w, h = map(float, parts[1:5])
                boxes.append((cls, x, y, w, h))
    except Exception:
        raise ValueError("Failed to parse YOLO label file")
    return boxes

# 生成三元组列表：(image_path, label_path_or_None, parsed_boxes_or_None)
image_label_pairs = []
test_images = [os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
for img in test_images:
    lp = label_path_for_image(img)
    parsed = parse_yolo_label(lp) if lp else None
    image_label_pairs.append((img, lp, parsed))

# 快速检查：总数与有标签的数量，显示前10项（每项显示图片、标签路径、box数）
total = len(image_label_pairs)
with_labels = sum(1 for _, lp, _ in image_label_pairs if lp)
print(f"total images: {total}, with label files: {with_labels}")

total images: 223, with label files: 223


In [7]:
for test_image_with_label in image_label_pairs:
    # test_image_with_label = image_label_pairs[210]

    img_path, _, parsed = test_image_with_label
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError("Could not load image")

    h, w = img.shape[:2]

    points = np.array([(x * w, y * h) for _, x, y, _, _ in parsed])


    # Parameters (adjust as needed; these are reasonable defaults for crowd detection)
    R = 80  # radius in pixels for density check
    MinPts = 3  # minimum points for core point
    ClusterSize_threshold = 5  # minimum cluster size for abnormal

    # Perform DBSCAN clustering
    dbscan = DBSCAN(eps=R, min_samples=MinPts)
    labels = dbscan.fit_predict(points)

    # Group points by cluster
    clusters = {}
    for i, label in enumerate(labels):
        if label not in clusters:
            clusters[label] = []
        clusters[label].append(points[i])

    # Identify abnormal clusters (exclude noise label -1)
    abnormal_clusters = [pts for label, pts in clusters.items() if label != -1 and len(pts) >= ClusterSize_threshold]

    # Collect abnormal points
    abnormal_points = set()
    for cluster in abnormal_clusters:
        abnormal_points.update(tuple(pt) for pt in cluster)

    # Draw abnormal cluster circles (yellow, expanded by 5%)
    for cluster in abnormal_clusters:
        pts = np.array(cluster, dtype=np.float32)
        (x, y), radius = cv2.minEnclosingCircle(pts)
        radius *= 1.05  # expand by 5%
        cv2.circle(img, (int(x), int(y)), int(radius), (0, 255, 255), 3)  # BGR: yellow

    # Draw points
    for pt in points:
        cx, cy = pt
        if tuple(pt) in abnormal_points:
            color = (0, 0, 255)  # red
        else:
            color = (0, 255, 0)  # green
        cv2.circle(img, (int(cx), int(cy)), 5, color, -1)  # filled

    # Save the result image
    result_path = os.path.join(RESULT_SAVE_DIR, os.path.basename(img_path))
    cv2.imwrite(result_path, img)

    # Display the image (convert BGR to RGB for matplotlib)
    # plt.figure(dpi = 300)
    # plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    # plt.axis('off')
    # plt.show()